In [2]:
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

label_list = ['hotel-general', 'bathroom-general', 'hotel-atmosphere', 'bathroom-atmosphere',
    'hotel-cleanliness', 'bathroom-cleanliness', 'hotel-facilities', 'bathroom-facilities',
    'hotel-location', 'bed-general', 'hotel-price', 'bed-cleanliness',
    'room-general', 'catering-general', 'room-atmosphere', 'catering-price',
    'room-cleanliness', 'parking', 'room-facilities', 'staff']

val_df = pd.read_csv('val_data.csv')

class ReviewDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts = df['text'].astype(str).tolist()
        self.labels = df[label_list].values.astype('float32')
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], add_special_tokens=True,
                             max_length=self.max_length, padding='max_length',
                             truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(self.labels[idx])}

tokenizer = AutoTokenizer.from_pretrained('ainniy/xlm-roberta-hotel-topics')
model = AutoModelForSequenceClassification.from_pretrained('ainniy/xlm-roberta-hotel-topics').to(device)
model.eval()
best_thresholds = np.load('best_thresholds.npy')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [3]:
from sklearn.metrics import classification_report

val_dataset = ReviewDataset(val_df, tokenizer, max_length=128)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

all_probs = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids, attention_mask=attention_mask)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(batch['labels'].numpy())

all_probs = np.concatenate(all_probs, axis=0)
all_labels = np.concatenate(all_labels, axis=0).astype(int)

predictions = (all_probs >= best_thresholds).astype(int)

report = classification_report(
    all_labels,
    predictions,
    target_names=label_list,
    output_dict=True,
    zero_division=0
)

df_report = pd.DataFrame(report).T
print(df_report[['precision', 'recall', 'f1-score']].round(3))

                      precision  recall  f1-score
hotel-general             0.515   0.714     0.598
bathroom-general          0.357   0.714     0.476
hotel-atmosphere          0.778   0.700     0.737
bathroom-atmosphere       0.423   0.917     0.579
hotel-cleanliness         0.769   0.714     0.741
bathroom-cleanliness      0.615   0.727     0.667
hotel-facilities          0.577   0.732     0.645
bathroom-facilities       0.857   0.800     0.828
hotel-location            0.955   0.955     0.955
bed-general               0.930   0.930     0.930
hotel-price               0.781   0.833     0.806
bed-cleanliness           0.600   0.500     0.545
room-general              0.667   0.692     0.679
catering-general          0.895   0.944     0.919
room-atmosphere           0.783   0.812     0.798
catering-price            1.000   0.600     0.750
room-cleanliness          0.840   0.724     0.778
parking                   0.857   1.000     0.923
room-facilities           0.855   0.883     0.869
